# S2 Pair Panel (1D)

Builds long-format **pair panels** for universes A (FX), B (crypto), and C (Asia equities).

**Run order:** execute `02_research/s2_coint/notebooks/hypothesis_tests/H-001_universes.ipynb` **first** so locked pair artifacts (`s2_pairs_{A,B,C}_1d.csv`) are up to date, then run this notebook **second**.

**Timing contract (S2 — not Default close-fill, not S1 trade-date):**

| Role | Timestamp |
|------|-----------|
| Features / signal / decision | Close of bar `t` (`close_y` / `close_x` drive β, spread, z, ADF, half-life) |
| Optional alt data | After close `t`, before open `t+1` (if known then) |
| Orders | Queued just before open `t+1` |
| Fill | Open of `t+1` on both legs (crypto / no auction: next bar's OHLCV open) |
| First PnL / HL stops | From open `t+1` onward (path-check high/low after entry) |

Do **not** put next-bar open on the signal row. Same causal lag applies at every bar size (1D / 4H / 1H — H-002).

**Panel columns:** `open_y/high_y/low_y/close_y` and `open_x/high_x/low_x/close_x` (no `price_y` / `price_x`). Hedge math uses closes only.

**Pair identity:** after Engle–Granger screening, `pair_id = ticker_y|ticker_x` where orientation follows the EG-chosen direction (lower p-value of `y~x` vs `x~y`).

**Candidates:** same-venue pairs only via `ticker_venue_key` / `iter_same_venue_pairs` in `data.processing.s2_universe` (e.g. `.HK` with `.HK`, `.T` with `.T`; no HK↔JP). Within-venue theme drops use the manual `KEEP_PAIR_IDS` keep-list.

**No global panel START_DATE:** each pair starts on the first date both legs have a valid close (unbalanced panel OK).

**Research IS (fixed calendar `T` per universe):**

| Universe | `RESEARCH_IS_END` |
|----------|-------------------|
| A (FX) | `2020-12-31` |
| B (crypto) | `2022-12-31` |
| C (Asia) | `2021-12-31` |

- **Discovery:** Engle–Granger + discovery half-life use only `date <= T` for that universe. Same `T` for the train parquet.
- **Ineligible:** mutual bars with `date <= T` fewer than `min(ols_window, 252)` — NaN metrics, not an EG fail; shown in diagnostics only; not locked / not re-screened on OOS.
- **EG fail:** eligible but `pvalue >=` threshold — also not locked / not re-screened on OOS.
- **Sealed OOS:** `date > T`. Never used to pick pairs. Later hypothesis WF uses the shared train panel (one sleeve calendar), not a per-pair loop.

**Panel z:** fixed `Z_WINDOW=60` (adaptive window is H-012, not used here). Rolling `half_life` uses `HL_WINDOW`.

**Outputs per universe** (`01_data/data_files/s2_coint/`):

| File | Contents |
|------|----------|
| `s2_pairs_{A,B,C}_1d.csv` | Locked pairs + IS discovery metrics |
| `s2_panel_{A,B,C}_1d_train.parquet` | Research IS (`date <= RESEARCH_IS_END`) |
| `s2_panel_{A,B,C}_1d_full.parquet` | Full sample census |

Regenerate train/full parquets after this contract / schema change.

Hypotheses: `02_research/s2_coint/s2_hypothesis_log.md`. Notes: `05_strategies/s2_coint/s2_algorithm_notes.md`.


## 0. Imports & Config

In [1]:
import os
import sys

import pandas as pd

ROOT = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(ROOT, "01_data", "ingestion")):
    parent = os.path.dirname(ROOT)
    if parent == ROOT:
        break
    ROOT = parent
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from data.ingestion.equity_fetcher import fetch_ohlcv
from data.processing.s2_coint_store import build_pair_panel, screen_pair_cointegration
from data.processing.s2_universe import (
    iter_same_venue_pairs,
    load_s2_universes,
    ticker_venue_key,
)

NOTEBOOK_DIR = os.path.abspath(os.getcwd())
UNIVERSES_CSV = os.path.join(NOTEBOOK_DIR, "s2_universes.csv")

UNIVERSE_LABELS = ("A", "B", "C")  # forex, crypto, asia

# Download floor only (not a pair-panel start). Pairs still begin at first mutual close.
FETCH_START = "1990-01-01"
END_DATE = None

# Pre-registered research-IS end (screen + train). Sealed OOS is after this date.
RESEARCH_IS_END = {
    "A": "2020-12-31",
    "B": "2022-12-31",
    "C": "2021-12-31",
}
COINT_PVALUE_THRESHOLD = 0.05
OLS_WINDOW = 252
Z_WINDOW = 60
HL_WINDOW = 252
INCLUDE_ADF_PVALUE = True
INCLUDE_VARIANCE_JUMP = True

# After first screen run, replace with the few pair_ids to keep (EG y|x form).
# Empty list => keep all pairs that pass the p-value filter for that universe.
KEEP_PAIR_IDS = {
    "A": [],
    "B": [],
    "C": [],
}


def pairs_csv_path(label: str) -> str:
    return os.path.join(NOTEBOOK_DIR, f"s2_pairs_{label}_1d.csv")


def panel_train_path(label: str) -> str:
    return os.path.join(NOTEBOOK_DIR, f"s2_panel_{label}_1d_train.parquet")


def panel_full_path(label: str) -> str:
    return os.path.join(NOTEBOOK_DIR, f"s2_panel_{label}_1d_full.parquet")


## 1. Load universes & fetch OHLCV

Universe C uses `isAsian=True` so `.HK` / `.T` suffixes are preserved. Store never fetches — only transforms. Keep full OHLC frames for `build_pair_panel`; closes Series views feed the EG screen.

In [3]:
universes = load_s2_universes(UNIVERSES_CSV)
assert len(universes) == 3

ohlc_by_universe: dict[str, dict[str, pd.DataFrame]] = {}
closes_by_universe: dict[str, dict[str, pd.Series]] = {}

for label, tickers in zip(UNIVERSE_LABELS, universes):
    is_asian = label == "C"
    ohlc: dict[str, pd.DataFrame] = {}
    closes: dict[str, pd.Series] = {}
    for ticker in tickers:
        panel = fetch_ohlcv(
            ticker,
            start_date=FETCH_START,
            end_date=END_DATE,
            isAsian=is_asian,
        )
        if panel is None or panel.empty:
            print(f"WARNING: no OHLCV for {ticker} (universe {label})")
            continue
        frame = (
            panel.sort_values("date")
            .drop_duplicates("date", keep="last")
            .set_index("date")[["open", "high", "low", "close"]]
            .astype(float)
        )
        frame.index = pd.to_datetime(frame.index)
        frame = frame.dropna(subset=["close"])
        if frame.empty:
            print(f"WARNING: no valid closes for {ticker} (universe {label})")
            continue
        ohlc[ticker] = frame
        closes[ticker] = frame["close"].copy()
        print(
            f"{label} {ticker:12} bars={len(frame):5d}  "
            f"venue={ticker_venue_key(ticker)}  "
            f"[{frame.index.min().date()} .. {frame.index.max().date()}]"
        )
    ohlc_by_universe[label] = ohlc
    closes_by_universe[label] = closes

A AUDUSD=X     bars= 5268  venue=FX  [2006-05-16 .. 2026-08-15]
A NZDUSD=X     bars= 5893  venue=FX  [2003-12-01 .. 2026-08-15]
A EURUSD=X     bars= 5891  venue=FX  [2003-12-01 .. 2026-08-14]
A GBPUSD=X     bars= 5903  venue=FX  [2003-12-01 .. 2026-08-14]
A USDCHF=X     bars= 5958  venue=FX  [2003-09-17 .. 2026-08-15]
A USDJPY=X     bars= 7726  venue=FX  [1996-10-30 .. 2026-08-15]
B BTC-USD      bars= 4351  venue=CRYPTO  [2014-09-17 .. 2026-08-15]
B ETH-USD      bars= 3202  venue=CRYPTO  [2017-11-09 .. 2026-08-15]
B SOL-USD      bars= 2319  venue=CRYPTO  [2020-04-10 .. 2026-08-15]
B BNB-USD      bars= 3202  venue=CRYPTO  [2017-11-09 .. 2026-08-15]
C 0700.HK      bars= 5472  venue=HK  [2004-06-16 .. 2026-08-14]
C 9988.HK      bars= 1651  venue=HK  [2019-11-26 .. 2026-08-14]
C 3690.HK      bars= 1941  venue=HK  [2018-09-20 .. 2026-08-14]
C 1810.HK      bars= 1994  venue=HK  [2018-07-09 .. 2026-08-14]
C 0939.HK      bars= 5127  venue=HK  [2005-10-27 .. 2026-08-14]
C 1398.HK      bars= 487

## 2. Research IS cutoff

Pre-registered calendar `RESEARCH_IS_END` per universe (no fraction of unique dates). Screen and train both use `date <= T`.


In [4]:
is_end_by_universe: dict[str, pd.Timestamp] = {}

for label in UNIVERSE_LABELS:
    if label not in RESEARCH_IS_END:
        raise KeyError(f"missing RESEARCH_IS_END for universe {label}")
    is_end = pd.Timestamp(RESEARCH_IS_END[label])
    is_end_by_universe[label] = is_end
    print(f"{label}: RESEARCH_IS_END={is_end.date()}")


A: RESEARCH_IS_END=2020-12-31
B: RESEARCH_IS_END=2022-12-31
C: RESEARCH_IS_END=2021-12-31


## 3. Screen cointegrated pairs (research IS)

Same-venue candidates only. EG + discovery half-life use `date <= RESEARCH_IS_END`. Ineligible rows (`eligible=False`) have too few IS bars — not EG fails. Display all screened rows and diagnostic counts, then keep `eligible & pvalue < threshold`.


In [5]:
screened_by_universe: dict[str, pd.DataFrame] = {}
passing_by_universe: dict[str, pd.DataFrame] = {}

for label, closes in closes_by_universe.items():
    candidates = iter_same_venue_pairs(list(closes.keys()))
    print(f"\n=== Universe {label}: {len(candidates)} same-venue candidates ===")
    screened = screen_pair_cointegration(
        closes,
        candidates,
        is_end=is_end_by_universe[label],
        ols_window=OLS_WINDOW,
        pvalue_threshold=COINT_PVALUE_THRESHOLD,
    )
    screened = screened.sort_values(
        ["eligible", "pvalue"], ascending=[False, True], kind="mergesort"
    ).reset_index(drop=True)
    screened_by_universe[label] = screened

    n_candidates = len(screened)
    n_ineligible = int((~screened["eligible"]).sum())
    eligible = screened.loc[screened["eligible"]].copy()
    n_eg_fail = int((eligible["pvalue"] >= COINT_PVALUE_THRESHOLD).sum())
    passing = eligible.loc[eligible["pvalue"] < COINT_PVALUE_THRESHOLD].copy()
    passing_by_universe[label] = passing
    n_pass = len(passing)

    display(screened)
    print(
        f"diagnostics: n_candidates={n_candidates}  n_ineligible={n_ineligible}  "
        f"n_eg_fail={n_eg_fail}  n_pass={n_pass}"
    )
    print(f"passing eligible & p < {COINT_PVALUE_THRESHOLD}: {n_pass} / {n_candidates}")



=== Universe A: 15 same-venue candidates ===


,pair_id,ticker_y,ticker_x,pvalue,tstat,discovery_half_life,n_is_bars,is_end,eligible
0,USDCHF=X|GBPUSD=X,USDCHF=X,GBPUSD=X,0.112303,-2.991236,31.478125,4439,2020-12-31,True
1,AUDUSD=X|USDJPY=X,AUDUSD=X,USDJPY=X,0.120759,-2.957097,33.484381,3790,2020-12-31,True
2,NZDUSD=X|USDCHF=X,NZDUSD=X,USDCHF=X,0.148225,-2.856986,37.448308,4428,2020-12-31,True
3,NZDUSD=X|USDJPY=X,NZDUSD=X,USDJPY=X,0.151982,-2.844345,36.576279,4415,2020-12-31,True
4,NZDUSD=X|AUDUSD=X,NZDUSD=X,AUDUSD=X,0.163407,-2.807191,41.860587,3806,2020-12-31,True
5,NZDUSD=X|GBPUSD=X,NZDUSD=X,GBPUSD=X,0.197847,-2.704827,48.948810,4427,2020-12-31,True
6,NZDUSD=X|EURUSD=X,NZDUSD=X,EURUSD=X,0.235013,-2.604519,32.023322,4414,2020-12-31,True
7,EURUSD=X|GBPUSD=X,EURUSD=X,GBPUSD=X,0.368465,-2.310219,25.958663,4425,2020-12-31,True
8,EURUSD=X|USDJPY=X,EURUSD=X,USDJPY=X,0.424514,-2.199708,38.222939,4429,2020-12-31,True
9,USDCHF=X|EURUSD=X,USDCHF=X,EURUSD=X,0.437666,-2.174275,15.888573,4427,2020-12-31,True


diagnostics: n_candidates=15  n_ineligible=0  n_eg_fail=15  n_pass=0
passing eligible & p < 0.05: 0 / 15

=== Universe B: 6 same-venue candidates ===


,pair_id,ticker_y,ticker_x,pvalue,tstat,discovery_half_life,n_is_bars,is_end,eligible
0,BNB-USD|ETH-USD,BNB-USD,ETH-USD,0.050159,-3.334543,64.766630,1879,2022-12-31,True
1,BNB-USD|BTC-USD,BNB-USD,BTC-USD,0.164710,-2.803069,115.065345,1879,2022-12-31,True
2,ETH-USD|SOL-USD,ETH-USD,SOL-USD,0.279796,-2.498354,39.896601,996,2022-12-31,True
3,BTC-USD|ETH-USD,BTC-USD,ETH-USD,0.355998,-2.335461,78.102926,1879,2022-12-31,True
4,BTC-USD|SOL-USD,BTC-USD,SOL-USD,0.713037,-1.618458,63.689650,996,2022-12-31,True
5,BNB-USD|SOL-USD,BNB-USD,SOL-USD,0.773883,-1.466308,51.244520,996,2022-12-31,True


diagnostics: n_candidates=6  n_ineligible=0  n_eg_fail=6  n_pass=0
passing eligible & p < 0.05: 0 / 6

=== Universe C: 30 same-venue candidates ===


,pair_id,ticker_y,ticker_x,pvalue,tstat,discovery_half_life,n_is_bars,is_end,eligible
0,1398.HK|0939.HK,1398.HK,0939.HK,0.000020,-5.471811,23.401167,3746,2021-12-31,True
1,0939.HK|0700.HK,0939.HK,0700.HK,0.000643,-4.666828,42.439328,3995,2021-12-31,True
2,1398.HK|0700.HK,1398.HK,0700.HK,0.002467,-4.310307,45.498468,3746,2021-12-31,True
3,0939.HK|1810.HK,0939.HK,1810.HK,0.010496,-3.882161,23.655484,862,2021-12-31,True
4,0939.HK|3690.HK,0939.HK,3690.HK,0.017891,-3.708088,22.922208,809,2021-12-31,True
5,1398.HK|1810.HK,1398.HK,1810.HK,0.038626,-3.434834,33.686305,862,2021-12-31,True
6,8306.T|8316.T,8306.T,8316.T,0.048277,-3.349500,25.301985,4016,2021-12-31,True
7,7735.T|6146.T,7735.T,6146.T,0.112794,-2.989204,57.033490,5254,2021-12-31,True
8,1398.HK|3690.HK,1398.HK,3690.HK,0.116436,-2.974321,30.358751,809,2021-12-31,True
9,0939.HK|9988.HK,0939.HK,9988.HK,0.117228,-2.971130,26.302903,519,2021-12-31,True


diagnostics: n_candidates=30  n_ineligible=0  n_eg_fail=23  n_pass=7
passing eligible & p < 0.05: 7 / 30


## 4. Lock pairs (`KEEP_PAIR_IDS`)

Venue keys already block cross-exchange pairs (e.g. HK↔JP). Use the keep-list to drop within-venue themes you do not want (e.g. HK tech vs HK banks). Leave a universe’s list empty to keep every p-value passer.

Fill `KEEP_PAIR_IDS` in §0 using `pair_id` values from the screen table (EG `y|x` form), then re-run from here.

In [ ]:
locked_meta_by_universe: dict[str, pd.DataFrame] = {}
locked_pairs_by_universe: dict[str, list[tuple[str, str]]] = {}

for label, passing in passing_by_universe.items():
    keep_ids = KEEP_PAIR_IDS.get(label) or []
    if keep_ids:
        locked = passing.loc[passing["pair_id"].isin(keep_ids)].copy()
        missing = sorted(set(keep_ids) - set(locked["pair_id"]))
        if missing:
            raise ValueError(
                f"universe {label}: KEEP_PAIR_IDS not in p-value passers: {missing}"
            )
    else:
        locked = passing.copy()
        print(f"{label}: KEEP_PAIR_IDS empty — keeping all {len(locked)} p-value passers")

    locked_meta_by_universe[label] = locked.reset_index(drop=True)
    locked_pairs_by_universe[label] = [
        (str(r.ticker_y), str(r.ticker_x))
        for r in locked.itertuples(index=False)
    ]
    print(f"{label}: locked {len(locked_pairs_by_universe[label])} pairs")
    display(locked_meta_by_universe[label])

## 5. Build full-history panels

Rolling OLS hedge, fixed-window z, rolling half-life on **closes only**; panel also stores open/high/low/close for both legs. Optional ADF / variance-jump columns.

In [ ]:
panels_by_universe: dict[str, pd.DataFrame] = {}

for label, pairs in locked_pairs_by_universe.items():
    if not pairs:
        print(f"{label}: no locked pairs — skipping panel build")
        panels_by_universe[label] = pd.DataFrame()
        continue
    panel = build_pair_panel(
        ohlc_by_universe[label],
        pairs,
        ols_window=OLS_WINDOW,
        z_window=Z_WINDOW,
        hl_window=HL_WINDOW,
        include_adf_pvalue=INCLUDE_ADF_PVALUE,
        include_variance_jump=INCLUDE_VARIANCE_JUMP,
    )
    panels_by_universe[label] = panel
    print(
        f"{label}: panel shape={panel.shape}  "
        f"pairs={panel['pair_id'].nunique() if not panel.empty else 0}  "
        f"dates=[{panel['date'].min().date() if not panel.empty else 'n/a'} .. "
        f"{panel['date'].max().date() if not panel.empty else 'n/a'}]"
    )
    display(panel.head())

## 6. Export pairs CSV + train / full parquets

Train panel: locked-pair rows with `date <= RESEARCH_IS_END` for that universe. Full panel is the census.


In [ ]:
for label in UNIVERSE_LABELS:
    meta = locked_meta_by_universe.get(label, pd.DataFrame())
    panel = panels_by_universe.get(label, pd.DataFrame())
    is_end = is_end_by_universe[label]

    meta_path = pairs_csv_path(label)
    meta.to_csv(meta_path, index=False)
    print(f"saved pairs metadata: {meta_path}  rows={len(meta)}")

    if panel.empty:
        print(f"{label}: empty panel — skip parquet export")
        continue

    dates = pd.to_datetime(panel["date"])
    full_panel = panel.copy()
    train_panel = panel.loc[dates <= is_end].copy()

    full_path = panel_full_path(label)
    train_path = panel_train_path(label)
    full_panel.to_parquet(full_path, index=False)
    train_panel.to_parquet(train_path, index=False)

    n_full_dates = dates.nunique()
    n_train_dates = (
        pd.to_datetime(train_panel["date"]).nunique() if not train_panel.empty else 0
    )
    print(f"saved full census: {full_path}  shape={full_panel.shape}")
    print(f"saved train / research IS: {train_path}  shape={train_panel.shape}")
    print(
        f"  RESEARCH_IS_END={is_end.date()}  "
        f"train_dates={n_train_dates}  full_dates={n_full_dates}"
    )
